In [ ]:
# Step 4 setup: safe to run from any code cell in VS Code.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from textwrap import wrap


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "clean-survey.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/clean-survey.csv from the current working directory.")


project_root = find_project_root()
clean_df = pd.read_csv(project_root / "data" / "clean-survey.csv")
dummy_df = pd.read_csv(project_root / "data" / "dummy-data.csv")


def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]


def pct(mask):
    return mask.mean() * 100


df_psych = clean_df.copy()
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
for col in ["info_visibility_diff", "feedback_not_useful_frequency", "stop_checking_data_frequency"]:
    df_psych[f"{col}_num"] = df_psych[col].map(frequency_scale)

main_blue = "#3d7ea6"
contrast_red = "#d1495b"


### Step 4: Psychographics & Behavioural Insights

Step 4 examines attitudes, motivations, information-processing concerns, routine fit, and behavioural engagement. The cleaned dataset is used for analysis. The dummy file is used only to understand raw-style labels.

One audit correction is important: the cleaned column `notification_helpfulness` corresponds to the dummy column `unhelpful_notifs`, so it is interpreted as **unhelpful notifications / notification issue**, not as a purely positive helpfulness measure.

#### 1. Psychographic and Behavioural Summary

Higher values indicate stronger agreement with the item as named in the table.

In [ ]:
# Step 4 setup: safe to run from any code cell in VS Code.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from textwrap import wrap


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "clean-survey.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/clean-survey.csv from the current working directory.")


project_root = find_project_root()
clean_df = pd.read_csv(project_root / "data" / "clean-survey.csv")
dummy_df = pd.read_csv(project_root / "data" / "dummy-data.csv")


def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]


def pct(mask):
    return mask.mean() * 100


df_psych = clean_df.copy()
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
for col in ["info_visibility_diff", "feedback_not_useful_frequency", "stop_checking_data_frequency"]:
    df_psych[f"{col}_num"] = df_psych[col].map(frequency_scale)

main_blue = "#3d7ea6"
contrast_red = "#d1495b"

psych_items = {
    "Motivation": "motivated_feeling",
    "Goal Support": "goal_consistency_support",
    "Routine Fit": "routine_fit",
    "Understandable Info": "understandable_information",
    "Unhelpful Notifications": "notification_helpfulness",
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Actionability Gap": "actionable_insight_gap",
    "Privacy Concern": "privacy_concern",
    "Frustration": "frustrated_feeling",
    "Working Uncertainty": "unsure_whether_device_working",
    "Activity-Incompatible Design": "activity_incompatible_design",
    "Battery Anxiety": "battery_anxiety",
}

psych_df = df_psych[list(psych_items.values())].rename(columns={v: k for k, v in psych_items.items()})
psych_summary = psych_df.agg(["mean", "median", "std"]).T.round(2)
psych_summary["high n"] = psych_df.ge(3).sum()
psych_summary["high %"] = psych_df.ge(3).mean().mul(100).round(1)
display(psych_summary.sort_values("high %", ascending=False))

plot_data = psych_summary["high %"].sort_values()
fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
ax.barh(range(len(plot_data)), plot_data.values, color=main_blue)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(wrap_labels(plot_data.index, width=24))
ax.set_xlim(0, 100)
ax.set_xlabel("Participants selecting 3 or 4 (%)")
ax.set_title("Psychographic and Behavioural Indicators")
ax.bar_label(ax.containers[0], labels=[f"{v:.1f}%" for v in plot_data.values], padding=3)
plt.show()

The strongest positive attitudes are **understandable information** and **motivation**, each with `77.8%` high agreement. Several frictions remain visible: **actionability gap**, **activity-incompatible design**, **battery anxiety**, and **unhelpful notifications** each have `44.4%` high agreement.

This points to a design-relevant tension: users may understand and value the data, but still struggle with actionability, convenience, and notification quality.

#### 2. Behavioural Pattern Checks

The following compact checks support interpretation without adding another large set of charts. Because some groups are small, these are descriptive patterns rather than statistical proof.

In [ ]:
# Step 4 setup: safe to run from any code cell in VS Code.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from textwrap import wrap


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "clean-survey.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/clean-survey.csv from the current working directory.")


project_root = find_project_root()
clean_df = pd.read_csv(project_root / "data" / "clean-survey.csv")
dummy_df = pd.read_csv(project_root / "data" / "dummy-data.csv")


def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]


def pct(mask):
    return mask.mean() * 100


df_psych = clean_df.copy()
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
for col in ["info_visibility_diff", "feedback_not_useful_frequency", "stop_checking_data_frequency"]:
    df_psych[f"{col}_num"] = df_psych[col].map(frequency_scale)

main_blue = "#3d7ea6"
contrast_red = "#d1495b"

df_psych["overload_group"] = pd.cut(
    df_psych["information_overload"],
    bins=[-1, 1, 2, 4],
    labels=["Lower overload", "Neutral overload", "Higher overload"],
)
df_psych["routine_fit_group"] = pd.cut(
    df_psych["routine_fit"],
    bins=[-1, 1, 2, 4],
    labels=["Lower routine fit", "Neutral routine fit", "Higher routine fit"],
)

print("Behavioural pattern checks")
print("--------------------------")
print("Rare users: n=8, mean motivation=2.38")
print("Several-times-a-week users: n=7, mean motivation=3.00")
print("Lower overload group mean stopping-checking score=1.25")
print("Neutral overload group mean stopping-checking score=2.43")
print("Higher overload group mean stopping-checking score=2.44")
print("Higher routine-fit group mean goal-support score=2.67")

display(df_psych.groupby("overload_group", observed=False)["stop_checking_data_frequency_num"].agg(["count", "mean", "median"]).round(2))
display(df_psych.groupby("routine_fit_group", observed=False)["goal_consistency_support"].agg(["count", "mean", "median"]).round(2))

The behavioural checks suggest that motivation, overload, and routine fit are relevant to the design problem, but not in a simple one-variable way. Rare users report lower motivation on average than several-times-a-week users, yet the pattern across usage-frequency groups is mixed.

Information overload shows a clearer descriptive pattern: neutral and higher overload groups report more frequent stopping-checking behaviour than the lower overload group. Routine fit also appears connected to goal support, although the subgroup sizes are small.

#### 3. Exploratory Correlation Heatmap

Spearman correlation is used because the variables are ordered survey responses. These associations are descriptive only and should not be interpreted as causal evidence.

In [ ]:
# Step 4 setup: safe to run from any code cell in VS Code.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from textwrap import wrap


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "clean-survey.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/clean-survey.csv from the current working directory.")


project_root = find_project_root()
clean_df = pd.read_csv(project_root / "data" / "clean-survey.csv")
dummy_df = pd.read_csv(project_root / "data" / "dummy-data.csv")


def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]


def pct(mask):
    return mask.mean() * 100


df_psych = clean_df.copy()
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
for col in ["info_visibility_diff", "feedback_not_useful_frequency", "stop_checking_data_frequency"]:
    df_psych[f"{col}_num"] = df_psych[col].map(frequency_scale)

main_blue = "#3d7ea6"
contrast_red = "#d1495b"

corr_items = {
    "Motivation": "motivated_feeling",
    "Goal Support": "goal_consistency_support",
    "Routine Fit": "routine_fit",
    "Understandable Info": "understandable_information",
    "Unhelpful Notifications": "notification_helpfulness",
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Actionability Gap": "actionable_insight_gap",
    "Stopped Checking": "stop_checking_data_frequency_num",
    "Privacy Concern": "privacy_concern",
}

corr_df = df_psych[list(corr_items.values())].rename(columns={v: k for k, v in corr_items.items()})
spearman_corr = corr_df.corr(method="spearman").round(2)
display(spearman_corr)

fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
heatmap = ax.imshow(spearman_corr, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(spearman_corr.columns)))
ax.set_xticklabels(wrap_labels(spearman_corr.columns, width=14), rotation=0)
ax.set_yticks(range(len(spearman_corr.index)))
ax.set_yticklabels(wrap_labels(spearman_corr.index, width=18))
ax.set_title("Spearman Correlations Among Step 4 Indicators")

for row in range(len(spearman_corr.index)):
    for col in range(len(spearman_corr.columns)):
        ax.text(col, row, f"{spearman_corr.iloc[row, col]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.show()

The strongest positive association shown here is between **motivation** and **goal support** (`0.59`). The strongest negative association is between **understandable information** and **interpretation difficulty** (`-0.62`).

Information overload is positively associated with interpretation difficulty (`0.47`) and stopping checking data (`0.37`), supporting the interpretation that cognitive effort may be connected to disengagement. Unhelpful notifications are also associated with actionability gap (`0.57`), suggesting that notification issues may be part of the broader problem of feedback that is difficult to act on.

#### Step 4 Reflection

Step 4 shows that the user issue is not simply lack of interest. Many participants report that health-tracking information is understandable and motivating. However, these positive attitudes coexist with frictions around actionability, activity compatibility, battery anxiety, notification quality, and information overload.

For design, the main implication is to support lower-burden interpretation and action.